## 1. Introduction

Many fundamental problems in algorithm design are **optimization problems**.  
An optimization problem asks us to **select the best solution** (maximum or minimum) from a set of feasible solutions, subject to one or more **constraints**.

Typical examples include:
- Maximizing profit under limited resources (e.g., Knapsack)
- Minimizing the number of coins needed to make change
- Determining whether a subset satisfies a target sum

A common characteristic of optimization problems is that:
- They can be naturally expressed using **recursive decision-making**
- Each step involves choosing among multiple alternatives

---

### Why Study Pure Recursion First?

Before introducing **Dynamic Programming (DP)**, it is essential to analyze the **naive recursive approach**.

Pure recursion:
- Provides a **clear and intuitive formulation**
- Closely mirrors the mathematical definition of the problem
- Helps identify **subproblems and decision structure**

However, despite its conceptual elegance, recursion often suffers from a severe limitation:

> **The same subproblems are solved repeatedly.**

This phenomenon is known as **redundant computation**.

---

### Consequence: Exponential Time Complexity

When recursive calls branch into multiple paths and overlap, the total number of function calls grows **exponentially** with input size. As a result:
- Small inputs may run quickly
- Slightly larger inputs become computationally infeasible

This explosive growth in running time is the key reason why **pure recursion is unsuitable for large optimization problems**.

---

### Scope of This Module

In this module, we analyze three classic optimization problems using **pure recursion**:

1. **0/1 Knapsack Problem**
2. **Coin Change Problem**
3. **Sum of Subsets Problem**

For each problem, we will:
- Write the recursive formulation
- Trace a numerical example
- Identify overlapping subproblems
- Prove why the time complexity becomes exponential

This analysis naturally motivates the transition to **Dynamic Programming**, where redundant computations are eliminated.


## 2. The 0/1 Knapsack Problem

### Problem Statement

You are given:
- **$n$** items
- Each item $i$ has:
  - **Weight:** $w_i$
  - **Value:** $v_i$
- A knapsack with **maximum capacity $W$**

**Goal:**  
Select a subset of items such that:
- The **total weight** does not exceed $W$
- The **total value** is maximized

⚠️ **Restriction (0/1 Property):**  
You cannot break items. Each item must be either:
- **Included (1)**, or
- **Excluded (0)**

---

<center>
    <img src="./img/Knapsack-Problem.png" alt="0/1 Knapsack Illustration">
</center>

---

### The Core Logic: *“Take It or Leave It”*

To understand recursion in the Knapsack problem, ignore formulas for a moment and think intuitively.

You inspect items **one by one**.  
For **each item**, you face a simple binary decision:

#### Step 1: Constraint Check — *Does it fit?*
- If the item’s weight is **greater than the remaining capacity**,  
  → You **must exclude** it.

#### Step 2: Decision — *Is it worth including?*
- If the item **does fit**, you explore **both possibilities**:
  - **Option A: Include the item**
    - Gain its value
    - Reduce remaining capacity
  - **Option B: Exclude the item**
    - Gain no value
    - Capacity remains unchanged

Since you don’t know which choice is better in advance, **recursion computes both outcomes** and selects the maximum.

---

### Numerical Example

Let us study the following concrete instance.

#### Constraints
- **Maximum Capacity (\(W\))**: \(15\ kg\)
- **Number of Items (\(n\))**: \(5\)

#### Item Table

| Item | Weight (\(w\)) | Value (\(v\)) |
|-----:|:--------------:|:-------------:|
| Item 1 | 12 kg | \$4 |
| Item 2 | 2 kg  | \$2 |
| Item 3 | 1 kg  | \$2 |
| Item 4 | 1 kg  | \$1 |
| Item 5 | 4 kg  | \$10 |

---

<center>
    <img src="./img/knapsack_recursion_tree.png" alt="Knapsack Recursion Tree">
</center>

---

### Visual Walkthrough of the Recursion Tree

Focus on the **first decision point** at **Item 5**.

#### Path 1: Include Item 5 (Green Path)
- Gain value: **\$10**
- Remaining capacity: **15 − 4 = 11 kg**
- Consequence:
  - Item 1 (12 kg) **no longer fits**
- **Total value:** \$10

#### Path 2: Exclude Item 5 (Red Path)
- Gain value: **\$0**
- Remaining capacity: **15 kg**
- Consequence:
  - Item 1 (12 kg) **fits**
- **Total value:** \$4

✅ **Decision:**  
The recursive algorithm compares both totals (\$10 vs \$4) and selects the **Include Item 5** branch.

---

### What Is a “Subproblem”?

Every recursive decision reduces the original problem into a **smaller instance of the same problem**.

- **Original Problem:**  
  Maximize value with **capacity 15** using **items 1–5**

- **Subproblem (after including Item 5):**  
  Maximize value with **capacity 11** using **items 1–4**

This smaller problem is solved using **exactly the same logic**, which is the essence of recursion.

---

### Recursive Formulation

For item index **n** and remaining capacity **W**:

1. **Base Case**
   - If `n == 0` or `W == 0`, no value can be added.

2. **Item Too Heavy**
   - If `weights[n-1] > W`, the item **must be excluded**.

3. **Item Fits**
   - Compute the maximum of:
     - **Include:**  
       `values[n-1] + Knapsack(n-1, W - weights[n-1])`
     - **Exclude:**  
       `Knapsack(n-1, W)`

---

### Pseudocode

```text
Algorithm Knapsack(n, W, weights, values):

1. Base Case:
   IF n == 0 OR W == 0:
       RETURN 0

2. If current item is too heavy:
   IF weights[n-1] > W:
       RETURN Knapsack(n-1, W, weights, values)

3. Otherwise:
   include = values[n-1] 
             + Knapsack(n-1, W - weights[n-1], weights, values)

   exclude = Knapsack(n-1, W, weights, values)

   RETURN MAX(include, exclude)
```

---
### Recursive Explosion and Overlapping Subproblems

Consider a slightly larger input.

The same recursive call (for example, `Knapsack(2, 20)`) can be reached:
- By **including** one particular combination of items
- By **excluding** a different combination of items

⚠️ **Key Observation:**  
The naive recursive algorithm does **not remember** results of previously solved subproblems. As a result, it recomputes the **same subproblem repeatedly**, wasting significant computation time.

---

### Time Complexity Analysis

- Each item introduces **two recursive branches**:
  - Include the item
  - Exclude the item
- The maximum depth of the recursion tree is **n** (one level per item)

**Time Complexity:**  
O(2^n)

This exponential growth makes the pure recursive approach impractical for large inputs and clearly motivates the need for **Dynamic Programming**.


### Recurrence Relation (Formal View)

Let `T(n, W)` denote the maximum value achievable using:
- the first **n** items, and
- a remaining knapsack capacity **W**

The recursive definition is:

- **Base Case**
  - If `n == 0` or `W == 0`:
    ```
    T(n, W) = 0
    ```

- **Item Too Heavy**
  - If `weights[n-1] > W`:
    ```
    T(n, W) = T(n-1, W)
    ```

- **Item Fits**
  - Otherwise:
    ```
    T(n, W) = max(
        values[n-1] + T(n-1, W - weights[n-1]),
        T(n-1, W)
    )
    ```

This recurrence directly captures the **“include or exclude”** decision at every step.



---

## 3. The Coin Change Problem

### Problem Statement

**The Challenge:**
You are given a set of coins of different denominations (via an array `coins`) and a total `amount`. The task is to find the **minimum number of coins** that add up to this amount.

<center>
    <img src="./img/coingchange.jpeg" alt="Coin Change Illustration">
</center>


**Assumptions:**

1. You have an **infinite supply** of each coin.
2. If the amount cannot be made up by any combination of the coins, the result is considered **Impossible**. By convention, we use **Infinity** to represent this state.

---

### Examples

Here are the standard cases we must handle. Note that we are only interested in the **count** of coins, not the specific list.

| Case | Input `coins` | Target `amount` | Output | Explanation |
| --- | --- | --- | --- | --- |
| **1** | `{2, 3, 5}` | `11` | **3** | Optimal: `5 + 3 + 3` |
| **2** | `{2, 3, 5, 7}` | `17` | **3** | Optimal: `7 + 7 + 3` |
| **3** | `{2, 3, 7}` | `15` | **4** | Optimal: `7 + 3 + 3 + 2` |
| **4** | `{3, 5}` | `7` | **Inf** | No combination adds to 7. |
| **5** | `{2, 3, 5}` | `1` | **Inf** | Smallest coin (2) is greater than amount. |

---

### Constraints & Complexity Goals

* **Coins Array Length:** 
* **Target Amount:** 
* **Values:** All denominations are non-zero positive integers.

**Target Efficiency (Dynamic Programming):**

* **Time Complexity:** .
* *Note: In the problem statement, this is often generalized as  where  represents the scale of inputs.*


* **Space Complexity:**  (Linear space).

---

### The Core Logic: *“Try Every Coin”*

Unlike the Knapsack problem where we have a simple "Include vs Exclude" choice, here we have a multi-way choice. For any current target `amount`, we don't know which coin leads to the optimal solution.

**The Strategy:**
At every step, we try **every valid coin** in our list and see which one yields the shortest path to 0.

#### The Recursive Formula

For a target sum :


**Handling Dead Ends:**

* If : We found a solution. Cost is **0**.
* If : We overshot. This path is **Invalid** (Infinity).

---

### Visual Walkthrough (Example 1)

Let's trace **Target 11** with Coins **{2, 3, 5}**.

1. **The Greedy Trap (Why we can't just pick the biggest coin):**
* If we pick **5** twice, we get . Remaining is **1**.
* We have no coin with value **1**.
* This path hits a **Dead End (Infinity)**.


2. **The Optimal Recursive Path:**
* The algorithm tries picking **5** once. Remaining: **6**.
* From 6, it tries picking **3**. Remaining: **3**.
* From 3, it picks **3**. Remaining: **0**.
* **Total Coins:** .



The recursion explores all valid combinations and guarantees finding the minimum.

---

### The Algorithm (Pseudocode)

We use a large number (like `Infinity`) to handle the "Not Possible" cases (Examples 4 & 5).

```text
Algorithm MinCoins(coins, amount):

    1. Base Case: Success
       IF amount == 0:
           RETURN 0
    
    2. Base Case: Failure (Impossible)
       IF amount < 0:
           RETURN Infinity

    3. Recursive Step:
       min_coins = Infinity
       
       FOR each coin 'c' in coins:
           # Recursive call: solve for the remainder
           res = MinCoins(coins, amount - c)
           
           # If the subproblem returned a valid number (not Infinity)
           IF res != Infinity:
               # Take the minimum of current best vs new result
               min_coins = MIN(min_coins, res + 1)

    4. Return Result
       RETURN min_coins

```

---

### Why Pure Recursion Fails

While correct, this approach is extremely slow for larger inputs.

* **Redundant Computation:** To solve for `11`, the algorithm might solve for `6` multiple times (once via , once via ).
* **Complexity:** Exponential .

**Next Step:**
To meet the expected time complexity of , we must use **Dynamic Programming (Memoization or Tabulation)** to store the results of subproblems so we never solve the same amount twice.

## 4. Coin Change: Number of Ways

### Problem Statement

**The Challenge:**
Given a set of coin denominations and a target `amount`, find the **total number of distinct combinations** of coins that sum up to the amount.

**Key Distinction:**

* **Order does not matter.**
* `{1, 2}` is the same combination as `{2, 1}`. You should count this only once.

### Examples

| Case | Coins | Amount | Output | Explanation |
| --- | --- | --- | --- | --- |
| **1** | `{1, 2, 3}` | `4` | **4** | `{1,1,1,1}`, `{1,1,2}`, `{1,3}`, `{2,2}` |
| **2** | `{2, 5, 3, 6}` | `10` | **5** | `{2,2,2,2,2}`, `{2,2,3,3}`, `{2,2,6}`, `{2,3,5}`, `{5,5}` |
| **3** | `{2}` | `3` | **0** | Cannot make 3 with only 2s. |

---

### The Core Logic: *“Include or Exclude”*

If we used the "Try Every Coin" logic (like in the Minimum Coins problem), we would overcount.

* *Path 1:* Pick 1, then Pick 2  `{1, 2}`
* *Path 2:* Pick 2, then Pick 1  `{2, 1}`
* *Result:* The algorithm counts this as **2 ways**, but it is actually **1 combination**.

To fix this, we switch to the **Knapsack Strategy** (Include/Exclude). We impose an order by processing coins one by one.

**For every coin type (index `n`), we make a choice:**

1. **Include it:** We use the coin. We reduce the amount, but we **stay at the same index `n**` (because we can use the same coin again).
2. **Exclude it:** We don't use this coin anymore. We **move to the next coin (index `n-1`)**.

**Recurrence Relation:**

<center>
    <img src="./img/coingchange.jpeg">
</center>


---

### Visual Walkthrough

Let's trace **Target 3** with Coins **{1, 2}**.

1. **Start at Coin 2:**
* **Choice A (Include 2):** Remainder is . We stay at Coin 2.
* Can we include 2 again? No ().
* We must **Exclude 2**  Move to Coin 1.
* Use Coin 1  Remainder 0. **(Found Way 1: `{2, 1}`)**


* **Choice B (Exclude 2):** Remainder is 3. Move to Coin 1.
* We must make 3 using only 1s.
* Use 1, 1, 1  Remainder 0. **(Found Way 2: `{1, 1, 1}`)**





**Total Ways:** .

---

### The Algorithm (Pseudocode)

```text
Algorithm CountWays(coins, n, amount):

    1. Base Case: Success
       IF amount == 0:
           RETURN 1  (Found exactly one valid way)

    2. Base Case: Failure
       IF amount < 0:
           RETURN 0  (Overshot)
       IF n <= 0:
           RETURN 0  (No coins left to make remaining sum)

    3. Recursive Step:
       
       # Option 1: Include current coin (coins[n-1])
       # Note: We pass 'n' again because supply is infinite
       include = CountWays(coins, n, amount - coins[n-1])

       # Option 2: Exclude current coin
       # Note: We pass 'n-1' to move to the next coin
       exclude = CountWays(coins, n-1, amount)

       # Total ways is sum of both possibilities
       RETURN include + exclude

```

### Complexity Warning

Just like the others, this pure recursive solution is exponential.

* **Time Complexity:** Exponential ( in terms of decision branches).
* **DP Fix:** By storing results in a table `Table[n][amount]`, we reduce this to .

# 5. Sum of Subsets Problem

### Problem Statement

<center>
    <img src="./img/subset.png">
</center>


**Definition:**
You are given an array of non-negative numbers and a value `sum`. You have to find out whether a **subset** of the given array is present whose sum is equal to the given value.

**What is a Subset?**
A subset is a set that contains elements of a previously defined set.
* Example: $\{a, b\}$ is a subset of $\{a, b, c, d\}$.
* **Goal:** Find a selection of numbers from the array that add up to the target `sum`.

---

### Examples

**Example 1:**
* **Input:** $\{10, 0, 5, 8, 6, 2, 4\}$, Target: $15$
* **Output:** `True`
* **Explanation:** The subset $\{5, 8, 2\}$ adds up to $5 + 8 + 2 = 15$.

**Example 2:**
* **Input:** $\{10, 0, 5, 8, 6, 2, 4\}$, Target: $3$
* **Output:** `False`
* **Explanation:** No combination of numbers adds up to 3 (assuming we cannot break numbers).

---

<center>
    <img src="./img/sunbset2.jpg">
</center>


---
### The Core Logic: *Recursion (Method 01)*

Recursion is ideal here because we need to make a sequence of decisions. For every element in the array, we have exactly two choices:

1.  **Include the element:**
    * We add it to our current subset.
    * The new target becomes $\text{sum} - \text{element}$.
2.  **Exclude the element:**
    * We skip it.
    * The target remains $\text{sum}$.

**The Trick:**
If **either** choice leads to a solution (returns `True`), then the answer for the current state is `True`. We combine these choices using the **OR** ($||$) operator.

---

### Visual Walkthrough

Let's visualize the "Include vs Exclude" logic with a small subset of the previous example.
* **Set:** $\{5, 8, 2\}$
* **Target:** $15$

**Step 1: Start with Target 15**
We look at the last item: **2**.
* **Option A (Include 2):** New Target is $15 - 2 = 13$.
* **Option B (Exclude 2):** Target remains $15$.

**Step 2: Follow Option A (Target 13)**
We look at the next item: **8**.
* **Option A (Include 8):** New Target is $13 - 8 = 5$.
* **Option B (Exclude 8):** Target remains $13$.

**Step 3: Follow Option A (Target 5)**
We look at the next item: **5**.
* **Option A (Include 5):** New Target is $5 - 5 = 0$.
* **RESULT:** Target reached **0**. Return `True`.

Since one path resulted in `True`, the entire problem returns `True`.

---

### Algorithm

1.  **Input:** Array `arr` and value `sum`.
2.  **Base Cases:**
    * If `sum == 0`: Return `True` (Solution found).
    * If `sum > 0` and no items left: Return `False`.
3.  **Recursive Step:**
    * Check if the last element is greater than `sum`.
        * **Yes:** Ignore it (Recurse for `n-1`, `sum`).
        * **No:** Check both possibilities:
            1.  **Include:** `isSubsetSum(arr, n-1, sum - arr[n-1])`
            2.  **Exclude:** `isSubsetSum(arr, n-1, sum)`
    * Return `Include || Exclude`.

---

### Recursive Formula

Let $\text{SubsetSum}(n, S)$ be the boolean function finding if sum $S$ is possible with first $n$ elements.

$$
\text{SubsetSum}(n, S) = \text{SubsetSum}(n-1, S) \lor \text{SubsetSum}(n-1, S - \text{arr}[n-1])
$$

**Base Cases:**
$$
\text{SubsetSum}(n, S) = 
\begin{cases} 
\text{True} & \text{if } S = 0 \\
\text{False} & \text{if } S > 0 \text{ and } n = 0 
\end{cases}
$$

---

### Pseudocode

```text
Algorithm isSubsetSum(arr, n, sum):

    1. Base Case (Found Solution):
       IF sum == 0:
           RETURN True

    2. Base Case (No items left):
       IF n == 0:
           RETURN False

    3. Optimization (Pruning):
       IF arr[n-1] > sum:
           # Element is too big, must exclude
           RETURN isSubsetSum(arr, n-1, sum)

    4. Recursive Step:
       # Check if sum can be obtained by including OR excluding
       is_included = isSubsetSum(arr, n-1, sum - arr[n-1])
       is_excluded = isSubsetSum(arr, n-1, sum)

       RETURN is_included OR is_excluded

```
### Complexity Analysis

* **Time Complexity:** $O(2^n)$
    * **Reasoning:** In the worst-case scenario (e.g., when no subset sums to the target or the target is very large), the recursion explores two branches (Include vs. Exclude) for every single element. This results in a binary tree of height $n$, leading to $2^n$ possible subsets.

* **Space Complexity:** $O(n)$
    * **Reasoning:** This is strictly for the recursion stack depth. The maximum depth of the recursion tree is $n$ (one call per element).

**Why Optimization is needed:**
For a small input like $n=30$, the operations explode to $2^{30}$, which is over **1 billion operations**. This exponential growth makes the naive recursive solution strictly unusable for large datasets, necessitating **Dynamic Programming** (Tabulation or Memoization) to solve it efficiently.